In [ ]:
from datasets import load_dataset
import pandas as pd
from tqdm import tqdm
import numpy as np
import re, math
import nltk
from nltk.tokenize import sent_tokenize

In [ ]:
from datasets import load_dataset
reviews = load_dataset("McAuley-Lab/Amazon-Reviews-2023",
                       "raw_review_All_Beauty",
                       trust_remote_code=True)


In [ ]:
import datasets
datasets.__version__

In [ ]:
meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    f"raw_meta_All_Beauty",
    split="full",
    trust_remote_code=True
)

In [ ]:
print("Reviews splits available:", reviews.keys())

In [ ]:
reviews["full"][1]

#asin : Amazon Standard Identification Number
# parent_asin : helps in joining reviews and meta data
#user_id : hashed amazon user id
#timestamp : review creation time
#helpful_vote : social signal / implicit relevance
#verified purchase : True if review is from a verified purchase

In [ ]:
meta[0]

#main_category : The high level amazon category the product belongs to
#average_rating : average rating of the product across all reviews
#rating_number : count of total ratings the product has received - helps in gauging popularity
#features : bullet points from the product listing
#description : product description text
#price : price of the product - string or numeric
# store: brand or seller of the product
# categories : hierarchical list of categories the product belongs to
# details : other product details like dimensions, weight, UPC: 12-digit barcode number used to uniquely identify a retail product in the U.S. and Canada. etc.
# parent_asin : for products that are variants (different size, color, etc.) of a main product, this field links to the main product's ASIN
# bought_together : list of ASINs that are frequently bought together with this product
# subtitles : list of available subtitles for the product listing



In [ ]:
reviews.shape, meta.shape

In [ ]:
split_ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    f"0core_timestamp_w_his_All_Beauty",
    trust_remote_code=True
)

In [ ]:
split_ds

In [ ]:
print({k: len(split_ds[k]) for k in split_ds.keys()})

In [ ]:
583190 + 71784 + 38955, reviews.shape

In [ ]:
for k in ["train", "valid", "test"]:
    print(k, split_ds[k][0])

In [ ]:
def hf_to_df(ds):
    # Turn an HF split into a pandas DataFrame
    return pd.DataFrame(ds)

def make_key(u, a, ts):
    # timestamps are strings in split files; standardize to str
    return f"{u}::{a}::{str(ts)}"

In [ ]:
reviews_full = reviews["full"]  # HF Dataset
keys = []
vals = []
for row in tqdm(reviews_full, desc="indexing raw reviews"):
    k = make_key(row.get("user_id"), row.get("parent_asin"), row.get("timestamp"))
    vals.append({
        "review_title": row.get("title"),
        "review_text": row.get("text"),
        "helpful_vote": row.get("helpful_vote"),
        "verified_purchase": row.get("verified_purchase"),
    })
    keys.append(k)

review_lookup = pd.DataFrame(vals, index=keys)

review_lookup.head()

In [ ]:
meta_df = pd.DataFrame(meta)
meta_lookup = meta_df.set_index("parent_asin")[[
    "main_category","title","average_rating","rating_number","price","store","categories","details"
]].rename(columns={"title":"item_title"})

meta_lookup.head()

In [ ]:
'''
Enrich a dataset split with review and metadata information.
'''
def enrich_split(split):
    """
    Join the split (HF dataset split) with:
      - review text fields from `review_lookup` (indexed by "user::parent::timestamp")
      - product meta from `meta_lookup` (indexed by parent_asin)
    WITHOUT changing any other helpers/globals.
    """
    import pandas as pd
    import numpy as np

    # --- 0) Materialize split
    df = hf_to_df(split).copy()

    # --- 1) Normalize split timestamp to SECONDS (handles ms vs s)
    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    if df["timestamp"].dropna().median() > 10**12:         # looks like ms
        df["timestamp"] = (df["timestamp"] // 1000).astype("Int64")
    else:
        df["timestamp"] = df["timestamp"].astype("Int64")

    # --- 2) Turn `review_lookup`'s composite index into 3 columns and dedupe
    # The index is "user_id::parent_asin::timestamp"
    idx_series = review_lookup.index.to_series().astype(str)
    keys = idx_series.str.split("::", n=2, expand=True)
    keys.columns = ["user_id", "parent_asin", "timestamp"]

    # Parse types
    keys["timestamp"] = pd.to_numeric(keys["timestamp"], errors="coerce")
    # Align units to SECONDS like df
    if keys["timestamp"].dropna().median() > 10**12:
        keys["timestamp"] = (keys["timestamp"] // 1000).astype("Int64")
    else:
        keys["timestamp"] = keys["timestamp"].astype("Int64")

    # Combine keys + values
    reviews_text = pd.concat(
        [keys.reset_index(drop=True), review_lookup.reset_index(drop=True)],
        axis=1
    )

    # Dedupe on the 3-key join (keep the "best" first occurrence)
    # Heuristics if present: higher helpful_vote, longer text
    if "helpful_vote" in reviews_text.columns:
        reviews_text["_helpful_vote"] = pd.to_numeric(reviews_text["helpful_vote"], errors="coerce").fillna(0)
    else:
        reviews_text["_helpful_vote"] = 0
    if "review_text" in reviews_text.columns:
        reviews_text["_text_len"] = reviews_text["review_text"].fillna("").str.len()
    else:
        # if your columns are still 'title'/'text' (pre-rename), fallback
        txtcol = "review_text" if "review_text" in reviews_text.columns else ("text" if "text" in reviews_text.columns else None)
        reviews_text["_text_len"] = reviews_text.get(txtcol, pd.Series([""]*len(reviews_text))).fillna("").astype(str).str.len()

    reviews_text = reviews_text.sort_values(
        ["user_id","parent_asin","timestamp","_helpful_vote","_text_len"],
        ascending=[True, True, True, False, False]
    ).drop_duplicates(subset=["user_id","parent_asin","timestamp"], keep="first").drop(columns=["_helpful_vote","_text_len"])

    # --- 3) Left-join text fields by the 3-key
    df = df.merge(
        reviews_text,
        on=["user_id","parent_asin","timestamp"],
        how="left",
        validate="m:1"  # split keys should be unique; text side deduped to 1
    )

    # --- 4) Left-join META by parent_asin
    # meta_lookup is already indexed by parent_asin
    df = df.join(meta_lookup, on="parent_asin", how="left")

    # --- 5) Final tidying
    if "rating" in df.columns:
        df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
    # ensure timestamp column stays integer-like
    df["timestamp"] = df["timestamp"].astype("Int64")

    return df

train_df = enrich_split(split_ds["train"])
valid_df = enrich_split(split_ds["valid"])
test_df = enrich_split(split_ds["test"])

for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    print(name, df.shape, list(df.columns)[:12])    
    print(df.head())

In [ ]:
train_df.head()

#### Aspect Tagging:

Aspect-based labeling or aspect extraction is the process of identifying which attribute or feature of a product a particular sentence or phrase is talking about.

In simple terms:
It tells what part of the product the sentence refers to — price, quality, delivery, durability, size, etc.

Example: 
| Review Sentence                           | Aspect Tag |
|------------------------------------------|-------------|
| “The delivery was super fast!”           | Delivery    |
| “Battery life is disappointing.”         | Durability  |
| “Totally worth the price.”               | Price       |
| “The screen size is perfect for travel.” | Size/Fit    |
| “Sound quality is crystal clear.”        | Quality     |


When you build personalized review summaries, different users value different aspects:
- One user cares about delivery speed
- Another cares only about durability
- A third focuses on price

So if your system knows which sentences belong to which aspect, it can:
- Filter reviews by user preference (“ignore delivery, show quality & durability”)
- Weight sentences differently in a personalized summary
- Aggregate ratings per aspect (e.g., “average quality rating = 4.7”)


In [ ]:
# This is kind of a bit manual aspects finding.

ASPECTS = {
    "delivery":  [r"\b(deliver(y|ed|ing)|ship(ping|ped)|arriv(ed|al))\b"],
    "price":     [r"\b(price|priced|expensive|cheap|value|worth|deal|cost)\b"],
    "quality":   [r"\b(quality|build|material|craftsmanship|well[- ]made|poorly[- ]made)\b"],
    "durability":[r"\b(durable|durability|last(ed|s)|broke|broken|sturdy|fragile)\b"],
    "fit":       [r"\b(size|fit|fitting|tight|loose|true to size)\b"],
}

ASPECTS

In [ ]:
# Compile regex patterns for efficiency
# Pre-compiles all regex patterns once for fast, repeated matching when tagging review sentences by aspect. 
aspect_regex = {a: [re.compile(p, flags=re.I) for p in pats] for a,pats in ASPECTS.items()}
aspect_regex


In [ ]:
# function that actually applies your regex aspect patterns to a given sentence and returns which aspects it mentions. 
def tag_sentence_aspects(sent):
    tags = {a: 0 for a in ASPECTS}
    for a, patterns in aspect_regex.items():
        if any(p.search(sent or "") for p in patterns):
            tags[a] = 1
    return tags

In [ ]:
import nltk
nltk.download('punkt') 
nltk.download('punkt_tab')
sent_tokenize("This should now work. Let's check!")

In [ ]:
# Convert each long review into multiple sentence-level rows,
# each annotated with which aspects are mentioned in that sentence.
def explode_snippets(df, max_sents=10):
    """Explode reviews into sentence-level snippets with aspect indicators."""
    rows = []
    for _, r in df.iterrows():
        text = r.get("review_text") or ""
        sents = sent_tokenize(text)[:max_sents] # only keeps 10 sentences per review
        for sid, s in enumerate(sents):
            tags = tag_sentence_aspects(s)
            rows.append({
                "user_id": r["user_id"],
                "parent_asin": r["parent_asin"],
                "timestamp": r["timestamp"],
                "rating": r["rating"],
                "sent_id": sid,
                "sentence": s,
                **tags,
                "helpful_vote": r.get("helpful_vote", 0),
                "verified_purchase": r.get("verified_purchase", False)
            })
    return pd.DataFrame(rows)

train_snip = explode_snippets(train_df)
valid_snip = explode_snippets(valid_df)
test_snip  = explode_snippets(test_df)

train_snip.head()


In [ ]:
user = "AGKHLEW2SOWHNMFQIJGBECAF7INQ"          # example user_id
ts = 1588615855        # example timestamp

train_snip.loc[
    (train_snip["user_id"] == user) &
    (train_snip["timestamp"] == ts)
]["sentence"]

The problem here is aspects can be different for different types of products, here since we are using ALL Beauty products:

Beauty-specific aspect schema (starter)
Use as candidate labels (you can merge/split later):
- scent_fragrance
- texture_consistency (creamy, oily, sticky, lightweight)
- longevity_wear_time (staying power, smudge, transfer)
- shade_color_match (undertone, oxidation)
- pigmentation_coverage (payoff, opacity, buildable)
- application_blendability (applicator, brush, streaks)
- finish_look (matte, dewy, glossy, natural)
- skin_hair_type_compat (oily/dry/sensitive/curly/fine)
- ingredients_irritation_allergy (alcohol, parabens, breakout)
- moisturization_hydration (drying, nourishing)
- absorption_residue (greasy, film, sticky)
- removal_cleanup (easy removal, stains)
- performance_effectiveness (works/doesn’t work)
- packaging (pump, leaks, broken seal)
- price_value
- delivery authenticity (shipping speed)

#### Lets use Transformers for aspect tagging:

First we see an example here: 

Each score represents the model’s confidence (probability) that the premise (your sentence) entails the hypothesis “This sentence is about <label>.”

So internally, for each label, the model evaluates:
- premise = “Smells amazing and lasts all day!”
- hypothesis = “This sentence is about scent_fragrance.”
- and predicts the probability that the premise entails that hypothesis.


In [ ]:
from transformers import pipeline, AutoTokenizer

MODEL = "typeform/distilbert-base-uncased-mnli"  # smaller, fast, reliable

tok = AutoTokenizer.from_pretrained(MODEL, revision="main")  # no trust_remote_code needed
clf = pipeline(
    "zero-shot-classification",
    model=MODEL,
    tokenizer=tok,
    # device_map="auto"  # uncomment if you have GPU
)

clf("Smells amazing and lasts all day!", 
    candidate_labels=["scent_fragrance","longevity_wear_time","price"], # this should not be fixed, clustering of the embeddings: aspects could be discovered dynamically for each cluster
    multi_label=True)

Because this is multi-label classification, not multi-class.
- Multi-class: only one label can be true → scores are softmaxed to sum to 1.
- Multi-label: many labels can be true → each score is an independent probability between 0 and 1.

Thus, the model doesn’t normalize across labels — each one is tested independently.

So:
- 0.75 → the model is ~75% confident the sentence talks about scent/fragrance.
- 0.44 → ~44% confident it also talks about longevity/wear time.
- 0.002 → ~0% confident it talks about price.

In [ ]:
CANDIDATE_ASPECTS = [
    "scent_fragrance",
    "texture_consistency",
    "longevity_wear_time",
    "shade_color_match",
    "pigmentation_coverage",
    "application_blendability",
    "finish_look",
    "skin_hair_type_compat",
    "ingredients_irritation_allergy",
    "moisturization_hydration",
    "absorption_residue",
    "removal_cleanup",
    "performance_effectiveness",
    "packaging",
    "price_value",
    "delivery_authenticity",
]

MODEL = "typeform/distilbert-base-uncased-mnli"  # or "typeform/distilbert-base-uncased-mnli" (faster)
tok = AutoTokenizer.from_pretrained(MODEL)
zshot = pipeline(
    "zero-shot-classification",
    model=MODEL,
    tokenizer=tok,
    # device_map="auto",   # uncomment if you have GPU
)

def zero_shot_tag_sentence(sent, aspects=CANDIDATE_ASPECTS, threshold=0.35):
    if not sent or not sent.strip():
        return {a: 0 for a in aspects}
    res = zshot(
        sent,
        candidate_labels=aspects,
        hypothesis_template="This sentence is about {}.",
        multi_label=True,
    )
    scores = dict(zip(res["labels"], res["scores"]))
    return {a: int(scores.get(a, 0.0) >= threshold) for a in aspects}

def zero_shot_tag_sentence_with_scores(sent, aspects=CANDIDATE_ASPECTS, threshold=0.35):
    if not sent or not sent.strip():
        flags = {a: 0 for a in aspects}
        probs = {f"{a}_score": 0.0 for a in aspects}
        return {**flags, **probs}
    res = zshot(
        sent,
        candidate_labels=aspects,
        hypothesis_template="This sentence is about {}.",
        multi_label=True,
    )
    scores = {lbl: sc for lbl, sc in zip(res["labels"], res["scores"])}
    flags = {a: int(scores.get(a, 0.0) >= threshold) for a in aspects}
    probs = {f"{a}_score": float(scores.get(a, 0.0)) for a in aspects}
    return {**flags, **probs}


In [ ]:
def explode_snippets(df, tagger, text_col="review_text", max_sents=8, df_name=""):
    """
    Expands each review into per-sentence rows with aspect tags.

    Args:
        df: DataFrame to process.
        tagger: callable(sentence) -> dict of tags.
        text_col: column name containing the review text.
        max_sents: number of sentences to keep per review.
        df_name: optional string name to show in tqdm bar.
    """
    rows = []
    iterator = tqdm(df.iterrows(), total=len(df), desc=f"Processing {df_name}")
    
    for _, r in iterator:
        text = (r.get(text_col) or "").strip()
        sents = sent_tokenize(text)[:max_sents]
        for sid, s in enumerate(sents):
            tags = tagger(s)
            rows.append({
                "user_id": r["user_id"],
                "parent_asin": r["parent_asin"],
                "timestamp": r["timestamp"],
                "rating": r.get("rating"),
                "sent_id": sid,
                "sentence": s,
                **tags,
            })
    return pd.DataFrame(rows)

In [ ]:
train_snip = explode_snippets(
    train_df,
    tagger=lambda s: zero_shot_tag_sentence(s, threshold=0.35),
    df_name="train_df"
)

valid_snip = explode_snippets(
    valid_df,
    tagger=lambda s: zero_shot_tag_sentence(s, threshold=0.35),
    df_name="valid_df"
)

test_snip = explode_snippets(
    test_df,
    tagger=lambda s: zero_shot_tag_sentence(s, threshold=0.35),
    df_name="test_df"
)
